In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit

from sklearn.linear_model import LogisticRegression

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [4]:
df = pd.read_csv("data/datav3.csv")
df.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [5]:
df.shape

(7519, 29)

# Independence of Observations

## Detection

### Repeated entities and Duplicated IDs

#### Duplicated IDs

In [6]:
df.duplicated().sum()

np.int64(35)

In [7]:
pd.set_option('display.max_rows', None)

In [8]:
exact_duplicates = df[df.duplicated(keep=False)]
exact_duplicates.sort_values(['customer_id', 'observation_month']).head(20)

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
5077,CUST00501,2025-08-01,B08,33,3078.55,36733.47,4,14,33.36,1.40,2,0,5.8,2,10,2,0,5,13,Mobile,East,Dark,Bachelor,Organic,CMP04,-0.3185,-0.9865,0,0
6080,CUST00501,2025-08-01,B08,33,3078.55,36733.47,4,14,33.36,1.40,2,0,5.8,2,10,2,0,5,13,Mobile,East,Dark,Bachelor,Organic,CMP04,-0.3185,-0.9865,0,0
731,CUST00605,2025-08-01,B13,62,5851.43,70200.80,11,40,17.22,7.82,2,0,19.5,1,13,5,1,6,8,Mobile,North,Dark,PhD,Organic,CMP02,-0.3049,-0.4817,1,0
999,CUST00605,2025-08-01,B13,62,5851.43,70200.80,11,40,17.22,7.82,2,0,19.5,1,13,5,1,6,8,Mobile,North,Dark,PhD,Organic,CMP02,-0.3049,-0.4817,1,0
5180,CUST00634,2026-01-01,B04,49,5360.73,64293.39,3,7,4.85,13.97,1,1,24.1,2,9,3,1,6,19,Mobile,Central,Dark,Master,Affiliate,CMP01,0.5543,0.4336,0,0
5830,CUST00634,2026-01-01,B04,49,5360.73,64293.39,3,7,4.85,13.97,1,1,24.1,2,9,3,1,6,19,Mobile,Central,Dark,Master,Affiliate,CMP01,0.5543,0.4336,0,0
2884,CUST00639,2026-01-01,B14,36,3496.38,42021.33,7,22,26.89,21.72,0,0,27.7,3,4,4,1,5,14,Mobile,North,Dark,High School,Referral,CMP01,-0.0494,0.0916,0,0
6333,CUST00639,2026-01-01,B14,36,3496.38,42021.33,7,22,26.89,21.72,0,0,27.7,3,4,4,1,5,14,Mobile,North,Dark,High School,Referral,CMP01,-0.0494,0.0916,0,0
1601,CUST00887,2026-02-01,B15,38,4849.39,58041.66,10,30,20.81,21.70,1,0,32.0,1,9,6,1,6,19,Mobile,North,Dark,Bachelor,Social,CMP02,0.5303,-0.0009,0,0
6975,CUST00887,2026-02-01,B15,38,4849.39,58041.66,10,30,20.81,21.70,1,0,32.0,1,9,6,1,6,19,Mobile,North,Dark,Bachelor,Social,CMP02,0.5303,-0.0009,0,0


they are exact duplicated columns, I will drop them

In [9]:
df_clean = df.drop_duplicates()

In [10]:
df_clean.shape

(7484, 29)

#### Repeated entities

In [11]:
customer_counts = df_clean['customer_id'].value_counts()
customer_counts.head()

customer_id
CUST03090    3
CUST02133    3
CUST01676    3
CUST03203    3
CUST02897    3
Name: count, dtype: int64

In [12]:
repeated_customers = customer_counts[customer_counts > 1]
len(repeated_customers)

1775

In [13]:
pd.set_option('display.max_rows', None)

In [14]:
repeated_customers_df =  df_clean[df_clean['customer_id'].isin(repeated_customers.index)].sort_values(by='customer_id')
repeated_customers_df.head(10)

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
630,CUST00001,2025-12-01,B17,40,14331.28,172408.24,6,21,32.13,10.34,1,0,8.2,2,12,4,1,5,8,Mobile,South,Dark,Master,Search,CMP01,0.4028,-0.3411,1,0
4953,CUST00001,2025-11-01,B17,40,14525.17,173291.39,9,31,23.24,10.14,1,0,47.9,1,13,4,3,7,18,Mobile,South,Dark,Master,Partner_Rare,CMP02,1.4404,0.7305,1,0
4195,CUST00003,2026-04-01,B15,31,2283.80,27365.68,7,24,31.83,11.54,1,0,44.1,2,9,6,1,6,11,Mobile,North,Light,High School,Social,CMP06,2.5045,-0.0049,0,0
6624,CUST00003,2026-05-01,B15,31,2359.29,28074.30,7,23,33.04,7.09,2,0,50.2,0,5,3,0,5,19,Mobile,North,Light,High School,Search,CMP01,-0.2593,-0.6711,1,0
3236,CUST00004,2026-02-01,B15,27,4735.69,56815.43,9,33,43.15,23.18,0,0,15.4,3,13,8,1,6,11,Mobile,North,Dark,Bachelor,Referral,CMP01,1.1225,-0.0007,0,0
5128,CUST00004,2026-03-01,B15,27,4576.81,54304.18,6,22,17.01,16.45,1,0,19.2,2,3,2,1,7,8,Mobile,North,Dark,Bachelor,Search,CMP33,0.7423,0.2380,0,0
7101,CUST00006,2025-06-01,B10,40,6764.86,80682.02,11,33,25.74,28.96,1,0,25.9,4,10,4,3,9,9,Desktop,East,Light,Bachelor,Social,CMP01,0.5939,0.2385,1,0
2215,CUST00006,2025-07-01,B10,40,6702.88,80338.78,8,31,25.39,27.59,0,0,35.9,4,12,4,0,3,12,Desktop,East,Light,Bachelor,Social,CMP03,-0.9011,-0.9781,0,0
3514,CUST00006,2025-05-01,B10,40,6758.65,81038.19,12,39,30.81,7.78,0,0,45.7,2,16,5,0,7,21,Desktop,East,Light,Bachelor,Search,CMP02,-0.4524,0.1382,0,0
1474,CUST00007,2026-02-01,B14,36,4494.10,54132.37,10,32,29.26,2.86,0,0,6.6,2,9,6,0,5,13,Mobile,North,Light,PhD,Affiliate,CMP08,-0.1623,0.3194,0,0


In [15]:
customer_counts.value_counts().sort_values()

count
3     509
2    1266
1    3425
Name: count, dtype: int64

In [16]:
df_clean[
    df_clean['customer_id'] == 'CUST00006'
][
    [
        'customer_id',
        'observation_month',
        'monthly_income',
        'purchased'
    ]
].sort_values('observation_month')

,customer_id,observation_month,monthly_income,purchased
3514,CUST00006,2025-05-01,6758.65,0
7101,CUST00006,2025-06-01,6764.86,0
2215,CUST00006,2025-07-01,6702.88,0


they are not duplicated rows, they are legitimate repeated measurements.Same customers with different time points.                                         
Treatments:
- `Agrregation` if data has multiple lower-level rows, but we want unit of analysis should be higher-level.
- `Group-aware train/test splitting`

### Natural groups or clusters

In [17]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


`1. Does the dataset have a meaningful grouping structure ?`

In [18]:
df_clean['branch_id'].nunique()

25

In [19]:
df_clean['branch_id'].value_counts().sort_values()

branch_id
B01    235
B05    252
B03    259
B15    264
B04    265
B02    268
B23    280
B25    282
B12    283
B22    293
B24    294
B09    296
B10    297
B14    297
B13    298
B11    302
B21    313
B08    316
B16    321
B19    321
B07    326
B17    343
B18    350
B06    353
B20    376
Name: count, dtype: int64

In [20]:
# number of unique customers in each branch
df.groupby('branch_id')['customer_id'].nunique().sort_values()

branch_id
B01    158
B04    173
B05    174
B03    177
B02    179
B15    189
B25    194
B23    196
B12    199
B10    203
B14    203
B11    203
B24    204
B22    210
B13    211
B09    211
B07    218
B19    219
B08    226
B21    227
B16    231
B06    241
B18    245
B17    247
B20    262
Name: customer_id, dtype: int64

the dataset has a meaningful grouping structure because multiple different customers belong to the same branch.

`2. Are observations within a group more similar than observations across groups?`

In [21]:
branch_summary = (
    df.groupby('branch_id')
      .agg(
          customers=('customer_id', 'nunique'),
          observations=('customer_id', 'size'),
          purchase_rate=('purchased', 'mean')
      )
)

branch_summary.sort_values(by='purchase_rate')

,customers,observations,purchase_rate
branch_id,,,
B11,203,303,0.075908
B17,247,344,0.110465
B14,203,299,0.120401
B09,211,298,0.120805
B18,245,352,0.122159
B10,203,299,0.127090
B15,189,267,0.127341
B12,199,284,0.130282
B04,173,267,0.138577


In [22]:
overall_purchase_rate = df['purchased'].mean()

overall_purchase_rate

np.float64(0.14509908232477722)

In [23]:
branch_rates = (
    df.groupby('branch_id')['purchased']
      .mean()
)

print("Min:", branch_rates.min())
print("Max:", branch_rates.max())
print("Std:", branch_rates.std())

Min: 0.07590759075907591
Max: 0.2361963190184049
Std: 0.029242373704262154


In [24]:
num_columns = ['age',
    'monthly_income', 
    'annual_income', 
    'website_visits_30d',
    'website_visits_90d', 
    'session_minutes', 
    'discount_pct',
    'support_tickets_30d', 
    'premium_member', 
    'days_since_last_purchase',
    'cart_items', 
    'email_opens_30d', 
    'profile_views_30d',
    'social_referrals_30d', 
    'survey_score', 
    'preferred_contact_hour',
    'noise_feature_1',
    'noise_feature_2', 
    'noise_feature_3', 
    'purchased']

In [25]:
braches_stats = df_clean.groupby('branch_id')[
    num_columns
].mean()

In [26]:
df_stats = df_clean[
    num_columns
].mean()


In [27]:
comparison_df = braches_stats.copy()
comparison_df.loc['overall_data'] = df_stats

In [28]:
comparison_df

,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,noise_feature_1,noise_feature_2,noise_feature_3,purchased
branch_id,,,,,,,,,,,,,,,,,,,,
B01,39.855319,5153.626213,61882.214298,7.851064,27.008511,21.443617,15.244638,0.731915,0.246809,38.020000,2.451064,9.085106,5.195745,1.310638,6.025532,15.421277,0.023607,-0.037803,0.510638,0.161702
B02,39.645522,5255.355112,63078.280709,7.936567,27.130597,20.868694,14.541306,0.839552,0.238806,37.321269,2.238806,8.996269,5.279851,1.205224,6.026119,15.328358,-0.012708,-0.028828,0.518657,0.141791
B03,41.146718,5589.874981,67060.601892,8.038610,27.486486,22.278880,15.066139,0.691120,0.351351,36.696525,2.552124,9.776062,5.100386,1.258687,6.057915,15.953668,0.078197,-0.005442,0.563707,0.162162
B04,39.169811,5092.008340,61126.968038,7.924528,27.079245,21.434377,15.006113,0.641509,0.267925,37.894340,2.384906,9.426415,5.354717,1.411321,5.939623,15.150943,0.008302,-0.005658,0.505660,0.139623
B05,39.539683,5457.503413,65454.897937,8.019841,27.563492,20.525516,15.074127,0.829365,0.253968,38.037698,2.567460,9.051587,5.396825,1.293651,5.988095,14.996032,-0.035949,-0.010597,0.511905,0.178571
B06,39.192635,5406.285779,64902.669547,7.903683,26.991501,21.730170,14.545552,0.796034,0.249292,37.288669,2.311615,9.546742,5.093484,1.311615,5.957507,14.920680,0.008227,-0.029516,0.501416,0.152975
B07,39.334356,5378.191994,64577.192730,8.024540,27.438650,21.629969,14.557239,0.812883,0.334356,35.618712,2.481595,9.389571,5.309816,1.340491,5.957055,15.273006,0.063428,0.020986,0.521472,0.236196
B08,38.936709,5665.908544,67987.409937,8.370253,28.382911,21.341519,14.467500,0.740506,0.319620,37.746203,2.455696,9.664557,5.582278,1.351266,6.022152,14.632911,-0.064384,-0.009787,0.537975,0.142405
B09,37.902027,5353.416689,64264.046182,7.908784,26.929054,21.541858,14.916520,0.824324,0.216216,38.088514,2.293919,9.138514,5.354730,1.337838,5.814189,14.293919,-0.053457,0.025434,0.425676,0.121622


In [29]:
braches_std = df_clean.groupby(by='branch_id')[
    num_columns
].std()

In [30]:
df_std = df_clean[
    num_columns
].std()


In [31]:
std_comparison_df = braches_std.copy()
std_comparison_df.loc['overall_data'] = df_std

std_comparison_df

,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,noise_feature_1,noise_feature_2,noise_feature_3,purchased
branch_id,,,,,,,,,,,,,,,,,,,,
B01,11.687725,2229.271651,26881.626595,2.799410,8.521462,10.358616,7.763017,0.862591,0.432075,24.811441,1.643671,3.554399,2.450358,1.254604,1.432001,5.749505,0.932206,0.583775,0.500954,0.368963
B02,11.741492,2108.244750,25340.724550,3.014274,8.810198,10.184501,7.323454,0.912052,0.427152,25.897699,1.553920,3.540560,2.451027,1.030822,1.402001,5.622688,0.964356,0.588585,0.500587,0.349488
B03,12.971708,2138.038198,25647.562071,3.093889,9.292971,11.293225,8.312282,0.847453,0.478317,28.881281,1.752126,3.307855,2.797694,1.154143,1.478701,5.644317,1.068789,0.610455,0.496885,0.369313
B04,10.932534,2162.973763,25869.365325,3.035454,9.268171,10.276355,7.763006,0.785627,0.443716,28.223239,1.681832,3.717697,2.589794,1.206306,1.399451,6.006298,0.956609,0.546553,0.500914,0.347251
B05,12.305637,2198.338703,26345.553461,2.916433,8.615280,10.104453,7.614628,0.860054,0.436146,25.686784,1.739915,3.548373,2.715103,1.129591,1.334442,5.752878,1.009056,0.614495,0.500853,0.383755
B06,11.089115,2024.682615,24245.785618,2.919724,8.593234,10.990163,7.694890,0.925206,0.433217,27.598156,1.703708,3.704834,2.756148,1.120220,1.506013,5.936161,0.905691,0.562244,0.500708,0.360474
B07,11.244493,2394.404663,28744.565581,3.220991,9.667117,11.123419,7.470921,0.879835,0.472490,25.648135,1.656121,3.686785,2.487945,1.143820,1.479500,5.413573,0.931035,0.587471,0.500307,0.425397
B08,11.666086,2476.571642,29706.660948,3.279412,9.520015,10.559021,7.442140,0.855832,0.467069,28.337786,1.765974,3.637740,2.658834,1.168379,1.424106,5.898996,1.029450,0.575209,0.499347,0.350019
B09,12.226417,2441.161169,29333.344376,3.187123,9.536995,9.743462,8.055469,0.869180,0.412361,26.413094,1.600243,3.847659,2.564261,1.173458,1.528021,5.888060,1.025707,0.564242,0.495282,0.327402


The dataset has a natural grouping structure through branch_id, but branch statistics do not vary strongly enough to suggest substantial branch-level dependence. Therefore, there is little evidence that the independence assumption is seriously violated because of branch clustering.|

### Time dependence / Autocorrelation

In [32]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [33]:
monthly_purchase = (
    df_clean
    .groupby('observation_month')['purchased']
    .mean()
    .sort_index()
)
monthly_purchase

observation_month
2025-01-01    0.165049
2025-02-01    0.169951
2025-03-01    0.187970
2025-04-01    0.156682
2025-05-01    0.157175
2025-06-01    0.153846
2025-07-01    0.114679
2025-08-01    0.126168
2025-09-01    0.126667
2025-10-01    0.103524
2025-11-01    0.119658
2025-12-01    0.144828
2026-01-01    0.136919
2026-02-01    0.149533
2026-03-01    0.165909
2026-04-01    0.133333
2026-05-01    0.160584
2026-06-01    0.159170
Name: purchased, dtype: float64

autocorrelation

In [34]:
for lag in range(1, 7):
    print(lag, monthly_purchase.autocorr(lag=lag))

1 0.5798520406363326
2 0.4216308923984829
3 0.41294853738540116
4 -0.2694053676743011
5 -0.5412611272856774
6 -0.52611500852856


The monthly purchase rate shows moderate positive autocorrelation over short lags, especially lag 1, suggesting nearby months are related. Negative correlations at lags 5–6 may indicate a cyclical pattern, but with only 18 months of data this should be treated as exploratory rather than strong evidence.

### Check for spatial dependence

In [35]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [36]:
df_clean['region'].value_counts()

region
South      1711
East       1588
West       1462
North      1444
Central    1279
Name: count, dtype: int64

In [37]:
df_clean.groupby(by='region')['purchased'].mean()

region
Central    0.156372
East       0.157431
North      0.119806
South      0.142607
West       0.149795
Name: purchased, dtype: float64

There is some regional difference in purchase rates, especially for the North region, but the differences are not large enough by themselves to indicate strong spatial dependence.

## Data Preparation

In [38]:
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0


In [39]:
df_clean.shape

(7484, 29)

In [40]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 7484 entries, 0 to 7518
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               7484 non-null   str    
 1   observation_month         7484 non-null   str    
 2   branch_id                 7484 non-null   str    
 3   age                       7484 non-null   int64  
 4   monthly_income            7484 non-null   float64
 5   annual_income             7484 non-null   float64
 6   website_visits_30d        7484 non-null   int64  
 7   website_visits_90d        7484 non-null   int64  
 8   session_minutes           7484 non-null   float64
 9   discount_pct              7484 non-null   float64
 10  support_tickets_30d       7484 non-null   int64  
 11  premium_member            7484 non-null   int64  
 12  days_since_last_purchase  7484 non-null   float64
 13  cart_items                7484 non-null   int64  
 14  email_opens_30d         

### Dummpy Encoding

In [41]:
cate_df = df_clean[['device_type', 'region', 'app_theme', 'marketing_channel']]
cate_df.head()

,device_type,region,app_theme,marketing_channel
0,Desktop,West,Dark,Organic
1,Mobile,Central,System,Email
2,Mobile,North,Light,Organic
3,Mobile,Central,Dark,Social
4,Desktop,South,Dark,Search


In [42]:
dumpy_encoded_df = pd.get_dummies(
    cate_df,
    drop_first=True,
    dtype=int
)
dumpy_encoded_df.head()

,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,1,0,0,0,0,0,0,0,1,0


In [43]:
df_clean = pd.concat([df_clean, dumpy_encoded_df], axis=1)
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,Bachelor,Organic,CMP60,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,Bachelor,Email,CMP02,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,Master,Organic,CMP04,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,Bachelor,Social,CMP10,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,Bachelor,Search,CMP01,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


### Ordinary Encoding

In [44]:
df_clean['education_level'].value_counts()

education_level
Bachelor       3125
High School    2081
Master         1761
PhD             517
Name: count, dtype: int64

In [45]:
education_level_mapping = {
    "High School":1,
    "Bachelor":2,
    "Master":3,
    "PhD":4
}
df_clean['education_level'] = df_clean['education_level'].map(education_level_mapping)
df_clean.head()

,customer_id,observation_month,branch_id,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,device_type,region,app_theme,education_level,marketing_channel,campaign_code,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST01723,2026-06-01,B24,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,Desktop,West,Dark,2,Organic,CMP60,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,CUST03090,2025-11-01,B05,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,Mobile,Central,System,2,Email,CMP02,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,CUST04684,2026-01-01,B13,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,Mobile,North,Light,3,Organic,CMP04,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,CUST01687,2025-03-01,B05,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,Mobile,Central,Dark,2,Social,CMP10,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CUST00535,2025-11-01,B17,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,Desktop,South,Dark,2,Search,CMP01,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


In [46]:
dropping_columns = ['device_type', 'region', 'app_theme', 'marketing_channel', 'branch_id', 'campaign_code']

df_clean = df_clean.drop(columns=dropping_columns)

In [47]:
df_ids = df_clean.copy()

In [48]:
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST01723,2026-06-01,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,CUST03090,2025-11-01,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,CUST04684,2026-01-01,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,CUST01687,2025-03-01,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CUST00535,2025-11-01,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


### Timestamp column

Observation-month data is  in timestamp format like '2022-02-08'. We need to convert it into numeric values: How many months have passed since the first date in the dataset. 

In [49]:
df_clean['observation_month'] = pd.to_datetime(
    df_clean['observation_month']
)

# getting the start date of data 
start_date = df_clean['observation_month'].min() 

# calculating 
df_clean['observation_month'] = (
    (df_clean['observation_month'].dt.year - start_date.year) * 12
    + 
    (df_clean['observation_month'].dt.month - start_date.month)
)
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


### Standardization

In [50]:
def z_score_stand(data):
    mean = data.mean()
    std = data.std()

    return (data - mean) / std

## Treatment

### Agrregation

When to apply this method: `if data has multiple lower-level rows, but we want unit of analysis should be higher-level.`
- now `one row = customer-month`, but we want to model `one row = customer=year`

In [51]:
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


In [52]:
df_ids['observation_month'] = pd.to_datetime(df_ids['observation_month'])
df_clean['year'] = df_ids['observation_month'].dt.year
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social,year
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,2026
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,2025
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,2026
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2025
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2025


In [53]:
customer_year_df = (
    df_clean
    .groupby(['customer_id', 'year'])
    .agg(
        age=('age', 'last'),

        monthly_income=('monthly_income', 'mean'),
        annual_income=('annual_income', 'mean'),

        website_visits_30d=('website_visits_30d', 'mean'),
        website_visits_90d=('website_visits_90d', 'mean'),

        session_minutes=('session_minutes', 'mean'),
        discount_pct=('discount_pct', 'mean'),
        support_tickets_30d=('support_tickets_30d', 'mean'),

        premium_member=('premium_member', 'last'),
        days_since_last_purchase=('days_since_last_purchase', 'last'),

        cart_items=('cart_items', 'mean'),
        email_opens_30d=('email_opens_30d', 'mean'),
        profile_views_30d=('profile_views_30d', 'mean'),
        social_referrals_30d=('social_referrals_30d', 'mean'),

        survey_score=('survey_score', 'mean'),
        preferred_contact_hour=('preferred_contact_hour', 'last'),

        education_level=('education_level', 'last'),

        noise_feature_1=('noise_feature_1', 'mean'),
        noise_feature_2=('noise_feature_2', 'mean'),
        noise_feature_3=('noise_feature_3', 'mean'),

        purchased=('purchased', 'max'),

        device_type_Mobile=('device_type_Mobile', 'last'),
        device_type_Tablet=('device_type_Tablet', 'last'),

        region_East=('region_East', 'last'),
        region_North=('region_North', 'last'),
        region_South=('region_South', 'last'),
        region_West=('region_West', 'last'),

        app_theme_Light=('app_theme_Light', 'last'),
        app_theme_System=('app_theme_System', 'last'),

        marketing_channel_Email=('marketing_channel_Email', 'last'),
        marketing_channel_Organic=('marketing_channel_Organic', 'last'),
        marketing_channel_Partner_Rare=('marketing_channel_Partner_Rare', 'last'),
        marketing_channel_Referral=('marketing_channel_Referral', 'last'),
        marketing_channel_Search=('marketing_channel_Search', 'last'),
        marketing_channel_Social=('marketing_channel_Social', 'last')
    )
    .reset_index()
)

In [54]:
customer_year_df.head()

,customer_id,year,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,CUST00001,2025,40,14428.225,172849.815,7.5,26.0,27.685,10.240,1.0,0,47.9,1.5,12.5,4.0,2.0,6.0,18,3,0.9216,0.19470,1.0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0
1,CUST00002,2025,49,4940.630,59718.200,6.0,23.0,20.980,22.280,0.0,0,26.6,0.0,4.0,1.0,2.0,4.0,17,3,1.0422,-0.83910,0.0,1,1,0,0,0,1,0,0,0,0,0,0,1,0,0
2,CUST00003,2026,31,2321.545,27719.990,7.0,23.5,32.435,9.315,1.5,0,50.2,1.0,7.0,4.5,0.5,5.5,19,1,1.1226,-0.33800,0.5,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0
3,CUST00004,2026,27,4656.250,55559.805,7.5,27.5,30.080,19.815,0.5,0,19.2,2.5,8.0,5.0,1.0,6.5,8,2,0.9324,0.11865,0.0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
4,CUST00005,2025,31,4199.810,50368.140,12.0,40.0,15.960,12.420,0.0,1,60.0,3.0,17.0,9.0,1.0,5.0,8,3,0.3404,0.60730,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [55]:
customer_year_df = customer_year_df.drop(columns=['customer_id', 'year'])

customer_year_df.head()

,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social
0,40,14428.225,172849.815,7.5,26.0,27.685,10.240,1.0,0,47.9,1.5,12.5,4.0,2.0,6.0,18,3,0.9216,0.19470,1.0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0
1,49,4940.630,59718.200,6.0,23.0,20.980,22.280,0.0,0,26.6,0.0,4.0,1.0,2.0,4.0,17,3,1.0422,-0.83910,0.0,1,1,0,0,0,1,0,0,0,0,0,0,1,0,0
2,31,2321.545,27719.990,7.0,23.5,32.435,9.315,1.5,0,50.2,1.0,7.0,4.5,0.5,5.5,19,1,1.1226,-0.33800,0.5,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0
3,27,4656.250,55559.805,7.5,27.5,30.080,19.815,0.5,0,19.2,2.5,8.0,5.0,1.0,6.5,8,2,0.9324,0.11865,0.0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
4,31,4199.810,50368.140,12.0,40.0,15.960,12.420,0.0,1,60.0,3.0,17.0,9.0,1.0,5.0,8,3,0.3404,0.60730,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


### Agrregation modeling

In [56]:
X = customer_year_df.drop(columns='purchased')
y = customer_year_df['purchased']

X_stand = z_score_stand(X)


In [57]:
def splitting_data(X, y, test_percentage=0.2):
    train_pc = 1 - test_percentage
    n = len(X)

    np.random.seed(42)
    indices = np.random.permutation(n)

    X_shuffled = X.iloc[indices]
    y_shuffled = y.iloc[indices] 

    X_train = X_shuffled[:int(train_pc * n)]
    y_train = y_shuffled[:int(train_pc * n)]

    X_test = X_shuffled[int(train_pc * n):]
    y_test = y_shuffled[int(train_pc * n):]

    return X_train, X_test, y_train, y_test 



In [58]:
X_train, X_test, y_train, y_test = splitting_data(X_stand, y)
print(f"Train data records {X_train.shape}")
print(f"Test data records: {X_test.shape}")

print(f"Train y: {y_train.shape}")
print(f"Test y: {y_test.shape}")

Train data records (4272, 34)
Test data records: (1068, 34)
Train y: (4272,)
Test y: (1068,)


In [59]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [60]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

confusion_matrix = pd.DataFrame(
    columns=['aggregated-data-model'],
    index=['Train_TP', "Train_TN", "Train_FP", "Train_FN", 'Test_TP', "Test_TN", "Test_FP", "Test_FN"]
)

train_TP = np.sum((train_predictions == 1) & (y_train == 1))
train_TN = np.sum((train_predictions == 0) & (y_train == 0))
train_FP = np.sum((train_predictions == 1) & (y_train == 0))
train_FN = np.sum((train_predictions == 0) & (y_train == 1))

test_TP = np.sum((test_predictions == 1) & (y_test == 1))
test_TN = np.sum((test_predictions == 0) & (y_test == 0))
test_FP = np.sum((test_predictions == 1) & (y_test == 0 ))
test_FN = np.sum((test_predictions == 0) & (y_test == 1))

confusion_matrix.loc['Train_TP'] = train_TP
confusion_matrix.loc['Train_TN'] = train_TN     
confusion_matrix.loc['Train_FP'] = train_FP
confusion_matrix.loc['Train_FN'] = train_FN

confusion_matrix.loc['Test_TP'] = test_TP
confusion_matrix.loc['Test_TN'] = test_TN
confusion_matrix.loc['Test_FP'] = test_FP
confusion_matrix.loc['Test_FN'] = test_FN

confusion_matrix

,aggregated-data-model
Train_TP,111
Train_TN,3400
Train_FP,74
Train_FN,687
Test_TP,26
Test_TN,841
Test_FP,21
Test_FN,180


In [61]:
models_restuls = pd.DataFrame(
    columns=['aggregated-data-model'],
    index=[
        'train-accuracy',
        'train-balanced-accuracy',
        'train-precision',
        'train-recall',
        'train-F1-score',
        'test-accuracy',
        'test-balanced-accuracy',
        'test-precision',
        'test-recall',
        'test-F1-score'
    ])


train_accuracy = (train_TP + train_TN) / (train_TP + train_TN + train_FP + train_FN)
train_balanced_acc = 1/2 * (train_TP / (train_TP + train_FN) + train_TN / (train_TN + train_FP))
train_precision = train_TP / (train_TP + train_FP)
train_recall = train_TP / (train_TP + train_FN)
train_F1_score = 2 *((train_precision * train_recall) / (train_precision + train_recall))

test_accuracy = (test_TP + test_TN) / (test_TP + test_TN + test_FP + test_FN)
test_balanced_acc = 1/2 * (test_TP / (test_TP + test_FN) + test_TN / (test_TN + test_FP))
test_precision = test_TP / (test_TP + test_FP)
test_recall = test_TP / (test_TP + test_FN)
test_F1_score = 2 * ((test_precision * test_recall) / (test_precision + test_recall))


models_restuls.loc['train-accuracy'] = train_accuracy
models_restuls.loc['train-balanced-accuracy'] = train_balanced_acc
models_restuls.loc['train-precision'] = train_precision
models_restuls.loc['train-recall'] = train_recall
models_restuls.loc['train-F1-score'] = train_F1_score
models_restuls.loc['test-accuracy'] = test_accuracy
models_restuls.loc['test-balanced-accuracy'] = test_balanced_acc
models_restuls.loc['test-precision'] = test_precision
models_restuls.loc['test-recall'] = test_recall
models_restuls.loc['test-F1-score'] = test_F1_score

models_restuls

,aggregated-data-model
train-accuracy,0.821863
train-balanced-accuracy,0.558898
train-precision,0.6
train-recall,0.139098
train-F1-score,0.225839
test-accuracy,0.811798
test-balanced-accuracy,0.550926
test-precision,0.553191
test-recall,0.126214
test-F1-score,0.205534


### Group-aware train-test split

In [62]:
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social,year
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,2026
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,2025
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,2026
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2025
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2025


In [63]:
df_clean.shape

(7484, 38)

In [64]:
groups = df_clean['customer_id']

X = df_clean.drop(columns=['customer_id', 'purchased'])
y = df_clean['purchased']



In [65]:
X_stand = z_score_stand(X)

In [66]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X_stand,
        y,
        groups=groups
    )
)

In [67]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [68]:
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

In [69]:
train_customers = set(groups_train)
test_customers = set(groups_test)

train_customers.intersection(test_customers)

set()

### Group-aware train-test split model

In [70]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

c:\Users\k\anaconda3\envs\creditrisk\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [71]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)


train_TP = np.sum((train_predictions == 1) & (y_train == 1))
train_TN = np.sum((train_predictions == 0) & (y_train == 0))
train_FP = np.sum((train_predictions == 1) & (y_train == 0))
train_FN = np.sum((train_predictions == 0) & (y_train == 1))

test_TP = np.sum((test_predictions == 1) & (y_test == 1))
test_TN = np.sum((test_predictions == 0) & (y_test == 0))
test_FP = np.sum((test_predictions == 1) & (y_test == 0 ))
test_FN = np.sum((test_predictions == 0) & (y_test == 1))

confusion_matrix['Group-aware-train-test-split-model'] = [
    train_TP,
    train_TN,
    train_FP,
    train_FN,
    test_TP,
    test_TN,
    test_FP,
    test_FN
]
confusion_matrix

,aggregated-data-model,Group-aware-train-test-split-model
Train_TP,111,80
Train_TN,3400,5077
Train_FP,74,51
Train_FN,687,777
Test_TP,26,15
Test_TN,841,1258
Test_FP,21,12
Test_FN,180,214


In [72]:
train_accuracy = (train_TP + train_TN) / (train_TP + train_TN + train_FP + train_FN)
train_balanced_acc = 1/2 * (train_TP / (train_TP + train_FN) + train_TN / (train_TN + train_FP))
train_precision = train_TP / (train_TP + train_FP)
train_recall = train_TP / (train_TP + train_FN)
train_F1_score = 2 *((train_precision * train_recall) / (train_precision + train_recall))

test_accuracy = (test_TP + test_TN) / (test_TP + test_TN + test_FP + test_FN)
test_balanced_acc = 1/2 * (test_TP / (test_TP + test_FN) + test_TN / (test_TN + test_FP))
test_precision = test_TP / (test_TP + test_FP)
test_recall = test_TP / (test_TP + test_FN)
test_F1_score = 2 * ((test_precision * test_recall) / (test_precision + test_recall))


models_restuls['Group-aware-train-test-split-model'] = [
    train_accuracy,
    train_balanced_acc,
    train_precision,
    train_recall,
    train_F1_score,
    test_accuracy,
    test_balanced_acc,
    test_precision,
    test_recall,
    test_F1_score
]

models_restuls

,aggregated-data-model,Group-aware-train-test-split-model
train-accuracy,0.821863,0.861654
train-balanced-accuracy,0.558898,0.541702
train-precision,0.6,0.610687
train-recall,0.139098,0.093349
train-F1-score,0.225839,0.161943
test-accuracy,0.811798,0.849233
test-balanced-accuracy,0.550926,0.528027
test-precision,0.553191,0.555556
test-recall,0.126214,0.065502
test-F1-score,0.205534,0.117188


### Time-based splitting

In [73]:
df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social,year
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,2026
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,2025
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,2026
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2025
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2025


In [74]:
df_clean['observation_month_date'] = df_ids['observation_month']

df_clean.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social,year,observation_month_date
0,CUST01723,17,57,4144.46,49861.80,8,25,18.74,9.01,3,1,16.1,2,11,2,1,5,17,2,-0.3758,0.7415,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,2026,2026-06-01
1,CUST03090,10,39,3313.13,39964.60,7,27,12.38,7.64,1,1,6.6,0,11,6,1,7,8,2,-0.6708,0.5766,1,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,2025,2025-11-01
2,CUST04684,12,51,4965.57,58703.65,3,15,8.97,15.84,3,0,55.0,0,3,5,1,4,8,3,0.2868,-0.6751,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,2026,2026-01-01
3,CUST01687,2,56,2045.66,24364.68,7,24,30.73,10.50,0,0,22.8,0,6,8,0,5,8,2,2.1992,0.8292,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2025,2025-03-01
4,CUST00535,10,31,4355.62,51882.27,10,29,31.40,2.09,0,0,40.1,1,10,4,0,8,8,2,-1.2592,-0.1662,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2025,2025-11-01


In [75]:
df_clean['observation_month_date'] = pd.to_datetime(
    df_clean['observation_month_date']
)

In [76]:
df_time = df_clean.sort_values(
    'observation_month'
).copy()

In [77]:
cutoff_date = pd.Timestamp('2026-03-01')


train_df = df_time[
    df_time['observation_month_date'] < cutoff_date
].copy()

test_df = df_time[
    df_time['observation_month_date'] >= cutoff_date
].copy()

In [78]:
train_df.head()

,customer_id,observation_month,age,monthly_income,annual_income,website_visits_30d,website_visits_90d,session_minutes,discount_pct,support_tickets_30d,premium_member,days_since_last_purchase,cart_items,email_opens_30d,profile_views_30d,social_referrals_30d,survey_score,preferred_contact_hour,education_level,noise_feature_1,noise_feature_2,noise_feature_3,purchased,device_type_Mobile,device_type_Tablet,region_East,region_North,region_South,region_West,app_theme_Light,app_theme_System,marketing_channel_Email,marketing_channel_Organic,marketing_channel_Partner_Rare,marketing_channel_Referral,marketing_channel_Search,marketing_channel_Social,year,observation_month_date
6258,CUST02076,0,54,9441.08,113268.12,12,42,38.90,31.94,0,1,24.4,5,12,7,1,6,18,3,-0.3522,-0.0737,1,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,2025,2025-01-01
6253,CUST01380,0,41,8044.30,96575.77,9,33,27.45,8.82,1,0,33.6,0,6,8,1,6,11,4,-0.3078,0.4938,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,2025,2025-01-01
2892,CUST01644,0,57,11312.53,136360.28,4,19,7.26,16.35,1,0,44.8,2,3,3,0,5,21,3,-0.0423,-0.0699,1,1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,2025,2025-01-01
2891,CUST04393,0,19,6204.29,74760.92,5,15,33.59,28.15,0,0,47.6,4,6,5,0,4,20,2,1.2994,-0.8592,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,2025,2025-01-01
4338,CUST04974,0,38,3353.98,40335.92,11,35,26.16,21.96,1,0,48.6,5,10,6,4,7,20,2,0.1685,-0.3385,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,1,2025,2025-01-01


In [87]:
train_df['observation_month_date'].value_counts().sort_index()

observation_month_date
2025-01-01    309
2025-02-01    406
2025-03-01    399
2025-04-01    434
2025-05-01    439
2025-06-01    429
2025-07-01    436
2025-08-01    428
2025-09-01    450
2025-10-01    454
2025-11-01    468
2025-12-01    435
2026-01-01    409
2026-02-01    428
Name: count, dtype: int64

In [86]:
test_df['observation_month_date'].value_counts().sort_index()

observation_month_date
2026-03-01    440
2026-04-01    420
2026-05-01    411
2026-06-01    289
Name: count, dtype: int64

In [92]:
X_train = train_df.drop(columns=['customer_id','year', 'purchased', 'observation_month_date'])
y_train = train_df['purchased']

X_test = test_df.drop(columns=['customer_id', 'year', 'purchased', 'observation_month_date'])
y_test = test_df['purchased']

X_train = z_score_stand(X_train)
X_test = z_score_stand(X_test)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

Train: (5924, 35)
Test: (1560, 35)


### Time-based split modelling

In [93]:
X_train = z_score_stand(X_train)
X_test = z_score_stand(X_test)

In [94]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [95]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)


train_TP = np.sum((train_predictions == 1) & (y_train == 1))
train_TN = np.sum((train_predictions == 0) & (y_train == 0))
train_FP = np.sum((train_predictions == 1) & (y_train == 0))
train_FN = np.sum((train_predictions == 0) & (y_train == 1))

test_TP = np.sum((test_predictions == 1) & (y_test == 1))
test_TN = np.sum((test_predictions == 0) & (y_test == 0))
test_FP = np.sum((test_predictions == 1) & (y_test == 0 ))
test_FN = np.sum((test_predictions == 0) & (y_test == 1))

confusion_matrix['Time-based-split-model'] = [
    train_TP,
    train_TN,
    train_FP,
    train_FN,
    test_TP,
    test_TN,
    test_FP,
    test_FN
]
confusion_matrix

,aggregated-data-model,Group-aware-train-test-split-model,Time-based-split-model
Train_TP,111,80,102
Train_TN,3400,5077,5013
Train_FP,74,51,66
Train_FN,687,777,743
Test_TP,26,15,21
Test_TN,841,1258,1302
Test_FP,21,12,17
Test_FN,180,214,220


In [96]:
train_accuracy = (train_TP + train_TN) / (train_TP + train_TN + train_FP + train_FN)
train_balanced_acc = 1/2 * (train_TP / (train_TP + train_FN) + train_TN / (train_TN + train_FP))
train_precision = train_TP / (train_TP + train_FP)
train_recall = train_TP / (train_TP + train_FN)
train_F1_score = 2 *((train_precision * train_recall) / (train_precision + train_recall))

test_accuracy = (test_TP + test_TN) / (test_TP + test_TN + test_FP + test_FN)
test_balanced_acc = 1/2 * (test_TP / (test_TP + test_FN) + test_TN / (test_TN + test_FP))
test_precision = test_TP / (test_TP + test_FP)
test_recall = test_TP / (test_TP + test_FN)
test_F1_score = 2 * ((test_precision * test_recall) / (test_precision + test_recall))


models_restuls['Time-based-split-model'] = [
    train_accuracy,
    train_balanced_acc,
    train_precision,
    train_recall,
    train_F1_score,
    test_accuracy,
    test_balanced_acc,
    test_precision,
    test_recall,
    test_F1_score
]

models_restuls

,aggregated-data-model,Group-aware-train-test-split-model,Time-based-split-model
train-accuracy,0.821863,0.861654,0.863437
train-balanced-accuracy,0.558898,0.541702,0.553858
train-precision,0.6,0.610687,0.607143
train-recall,0.139098,0.093349,0.120710
train-F1-score,0.225839,0.161943,0.201382
test-accuracy,0.811798,0.849233,0.848077
test-balanced-accuracy,0.550926,0.528027,0.537124
test-precision,0.553191,0.555556,0.552632
test-recall,0.126214,0.065502,0.087137
test-F1-score,0.205534,0.117188,0.150538
